In [6]:
!pip install --upgrade onnx
!pip install --upgrade onnx onnxruntime
!pip install --upgrade onnx onnx-simplifier
!pip install --upgrade onnx onnx-simplifier onnxruntime


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
# %%
import os
import numpy as np
import onnx
import onnxruntime
from onnxsim import simplify
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat, quant_pre_process
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat
from PIL import Image
from torchvision import transforms


In [8]:
# %%
# --- CONFIGURATION ---
BASE_DIR = os.getcwd()

# Chemins adaptés à votre projet YOLOP
INPUT_MODEL_PATH = os.path.join(BASE_DIR, "weights", "yolop-640-640.onnx")
OUTPUT_DIR = os.path.join(BASE_DIR, "weights")
OUTPUT_MODEL_PATH = os.path.join(OUTPUT_DIR, "yolop-640-640-quantized-per_channel.onnx")

# Utilisation du dossier de validation pour la calibration (images réelles)
CALIBRATION_IMG_DIR = os.path.join(BASE_DIR, "dataset", "images", "val")
CALIBRATION_SIZE = 100  # Nombre d'images pour calibrer (100-200 est suffisant)
TARGET_SIZE = (640, 640) # Taille d'entrée YOLOP


In [9]:
# %%

class YOLOPDataReader(CalibrationDataReader):
    def __init__(self, image_folder, model_path, size_limit=100):
        self.image_folder = image_folder
        self.enum_data_dicts = None
        self.datasize = 0
        
        # Récupération du nom de l'entrée du modèle (ex: 'images')
        session = onnxruntime.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        self.input_name = session.get_inputs()[0].name
        
        # Normalisation standard (ImageNet) utilisée par YOLOP
        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        files = [f for f in os.listdir(image_folder) if f.endswith('.jpg')]
        self.image_files = files[:size_limit]
        self.datasize = len(self.image_files)
        print(f"-> DataReader initialisé avec {self.datasize} images pour la calibration.")

    def letterbox_image(self, image, size):
        '''Redimensionne l'image en gardant le ratio (Letterbox)'''
        iw, ih = image.size
        w, h = size
        scale = min(w/iw, h/ih)
        nw = int(iw*scale)
        nh = int(ih*scale)

        image = image.resize((nw,nh), Image.BICUBIC)
        new_image = Image.new('RGB', size, (114,114,114)) # Fond gris (114) comme YOLOP
        new_image.paste(image, ((w-nw)//2, (h-nh)//2))
        return new_image

    def get_next(self):
        if self.enum_data_dicts is None:
            iter_data = iter(self.load_batch())
            self.enum_data_dicts = iter_data
        try:
            return next(self.enum_data_dicts)
        except StopIteration:
            return None

    def load_batch(self):
        for filename in self.image_files:
            image_path = os.path.join(self.image_folder, filename)
            image = Image.open(image_path).convert("RGB")
            
            # Application du Letterbox (Crucial pour YOLOP)
            image = self.letterbox_image(image, TARGET_SIZE)
            
            # Normalisation et conversion en numpy
            input_tensor = self.normalize(image)
            input_numpy = input_tensor.unsqueeze(0).numpy()
            
            yield {self.input_name: input_numpy}



In [ ]:
# %%

def quantize_model():
    global INPUT_MODEL_PATH
    print("========================================")
    print("   QUANTIFICATION YOLOP (STATIC PTQ)    ")
    print("========================================")
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if not os.path.exists(INPUT_MODEL_PATH):
        print(f"ERREUR : Modèle introuvable : {INPUT_MODEL_PATH}")
        return


    # --- PIPELINE DE NETTOYAGE (Simplification + Pré-traitement) ---
    print("----------------------------------------")
    print(f"NETTOYAGE DU MODELE (Simplification -> Pré-traitement)")
    
    # Étape 1 : Simplification (Fusion Conv+BN)
    print(f"1. Simplification onnx-simplifier...")
    SIMPLIFIED_PATH = os.path.join(OUTPUT_DIR, "temp_simplified.onnx")
    model_proto = onnx.load(INPUT_MODEL_PATH)
    model_sim, check = simplify(model_proto)
    if not check:
        print("❌ Echec de la simplification.")
        return
    onnx.save(model_sim, SIMPLIFIED_PATH)
    
    # Étape 2 : Pré-traitement ONNX Runtime (Shape Inference + Corrections)
    print(f"2. Pré-traitement quant_pre_process...")
    PREPROCESSED_PATH = os.path.join(OUTPUT_DIR, "yolop_clean_for_quant.onnx")
    quant_pre_process(SIMPLIFIED_PATH, PREPROCESSED_PATH)
    
    print(f"✅ Modèle prêt : {PREPROCESSED_PATH}")
    
    # Mise à jour du chemin d'entrée pour la suite
    INPUT_MODEL_PATH = PREPROCESSED_PATH
    print("----------------------------------------")

    


    # Initialisation du DataReader
    print(f"Préparation des données de calibration depuis : {CALIBRATION_IMG_DIR}")
    dr = YOLOPDataReader(CALIBRATION_IMG_DIR, INPUT_MODEL_PATH, size_limit=CALIBRATION_SIZE)
    
    print(f"Lancement de la quantification...")
    
    # Quantification
    # Note: On utilise QDQ (Quantize-DeQuantize) qui est le standard pour CPU/TensorRT
    quantize_static(
        model_input=INPUT_MODEL_PATH,
        model_output=OUTPUT_MODEL_PATH,
        calibration_data_reader=dr,
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QUInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,    
        reduce_range=True   
    )
    
    print("\nCalcul de la réduction...")
    if os.path.exists(OUTPUT_MODEL_PATH):
        size_original = os.path.getsize(INPUT_MODEL_PATH) / (1024 * 1024)
        size_quantized = os.path.getsize(OUTPUT_MODEL_PATH) / (1024 * 1024)
        
        print(f"   Original (FP32) : {size_original:.2f} MB")
        print(f"   Quantized (INT8): {size_quantized:.2f} MB")
        print(f"   Ratio           : {size_original / size_quantized:.2f}x plus petit")
        print(f"\n[OK] Modèle sauvegardé : {OUTPUT_MODEL_PATH}")
    else:
        print("Erreur : Le fichier de sortie n'a pas été créé.")


In [11]:
# %%

if __name__ == "__main__":
    quantize_model()

   QUANTIFICATION YOLOP (STATIC PTQ)    
----------------------------------------
PRE-PROCESSING ONNX (Correction des axes et fusion)
Pré-traitement en cours...
✅ Modèle pré-traité sauvegardé : /workspace/weights/yolop_preprocessed.onnx
----------------------------------------
Préparation des données de calibration depuis : /workspace/dataset/images/val
-> DataReader initialisé avec 100 images pour la calibration.
Lancement de la quantification...



Calcul de la réduction...
   Original (FP32) : 34.24 MB
   Quantized (INT8): 11.83 MB
   Ratio           : 2.89x plus petit

[OK] Modèle sauvegardé : /workspace/weights/yolop-640-640-quantized-per_channel.onnx
